In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("SALib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "SALib"])

In [3]:
import numpy as np
from scipy.integrate import solve_ivp
from SALib.sample import saltelli
from SALib.analyze import sobol

# =========================================================
# Duffing system
# =========================================================
def duffing(t, y, zeta, alpha, F, omega):
	x, v = y
	dxdt = v
	dvdt = -2*zeta*v - x - alpha*x**3 + F*np.cos(omega*t)
	return [dxdt, dvdt]

# =========================================================
# Simulation function
# =========================================================
def simulate(params):
	alpha, zeta, F = params
	omega = 1.2  # fixed forcing frequency

	t_span = (0, 200)
	t_eval = np.linspace(*t_span, 5000)

	sol = solve_ivp(
		duffing,
		t_span,
		[0, 0],
		t_eval=t_eval,
		args=(zeta, alpha, F, omega),
		method='RK45'
	)

	x = sol.y[0]

	# discard transient (last 30%)
	x_ss = x[int(0.7 * len(x)):]

	# output metric
	y = np.max(np.abs(x_ss))

	return y

# =========================================================
# SALib problem definition
# =========================================================
problem = {
	'num_vars': 3,
	'names': ['alpha', 'zeta', 'F'],
	'bounds': [
		[0.0, 5.0],    # alpha
		[0.01, 0.2],   # damping
		[0.1, 1.0]     # forcing
	]
}

# =========================================================
# Generate samples
# =========================================================
N = 512  # base sample size
param_values = saltelli.sample(problem, N, calc_second_order=True)

# =========================================================
# Run simulations
# =========================================================
Y = np.zeros(len(param_values))

for i, params in enumerate(param_values):
	if i % 100 == 0:
		print(f"Running {i}/{len(param_values)}")
	Y[i] = simulate(params)

# =========================================================
# Sobol analysis
# =========================================================
Si = sobol.analyze(problem, Y, calc_second_order=True, print_to_console=True)

# =========================================================
# Print results clearly
# =========================================================
print("\nFirst-order indices:")
for name, val in zip(problem['names'], Si['S1']):
	print(f"{name}: {val:.3f}")

print("\nTotal-order indices:")
for name, val in zip(problem['names'], Si['ST']):
	print(f"{name}: {val:.3f}")

C:\Users\setemadi3\AppData\Local\Temp\ipykernel_12040\3417348330.py:61: DeprecationWarning: `salib.sample.saltelli` will be removed in SALib 1.5.1 Please use `salib.sample.sobol`
  param_values = saltelli.sample(problem, N, calc_second_order=True)


Running 0/4096
Running 100/4096
Running 200/4096
Running 300/4096
Running 400/4096
Running 500/4096
Running 600/4096
Running 700/4096
Running 800/4096
Running 900/4096
Running 1000/4096
Running 1100/4096
Running 1200/4096
Running 1300/4096
Running 1400/4096
Running 1500/4096
Running 1600/4096
Running 1700/4096
Running 1800/4096
Running 1900/4096
Running 2000/4096
Running 2100/4096
Running 2200/4096
Running 2300/4096
Running 2400/4096
Running 2500/4096
Running 2600/4096
Running 2700/4096
Running 2800/4096
Running 2900/4096
Running 3000/4096
Running 3100/4096
Running 3200/4096
Running 3300/4096
Running 3400/4096
Running 3500/4096
Running 3600/4096
Running 3700/4096
Running 3800/4096
Running 3900/4096
Running 4000/4096
             ST   ST_conf
alpha  0.707829  0.108342
zeta   0.181396  0.159308
F      0.412447  0.154754
             S1   S1_conf
alpha  0.539478  0.238861
zeta   0.036078  0.040357
F      0.238670  0.076413
                     S2   S2_conf
(alpha, zeta) -0.051879  0.42200